In [ ]:
"""
========================================================================
 MODEL COMPLEXITY COMPARISON SCRIPT
 Computes: Params, Trainable Params, FLOPs, Model Size, Inference Time,
           Throughput  -- for every baseline + the proposed model.
 (No accuracy column -- pure complexity/efficiency analysis)

 Run this on Kaggle. Output -> model_complexity_comparison.csv
========================================================================
"""

import os
import gc
import time
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model

from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

print("TensorFlow:", tf.__version__)

# ============================ CONFIG ============================
IMG_SIZE     = 224
NUM_CLASSES  = 5          # <-- change to your actual number of classes
N_WARMUP     = 5          # warm-up runs before timing (excluded from avg)
N_TIMED_RUNS = 50         # runs actually used to measure inference time
BATCH_SIZE   = 1          # per-image latency; change to measure batch throughput
CSV_PATH     = "sabbir-final-csv-modelcomplexity.csv"
TMP_SAVE_DIR = "tmp_model_check"

result_columns = [
    "Model", "Total_Params_M", "Trainable_Params_M", "Non_Trainable_Params_M",
    "FLOPs_G", "Model_Size_MB", "Inference_Time_ms", "Throughput_img_per_sec"
]

# ============================ FLOPs COUNTER ============================
def count_flops(model, input_shape=(1, IMG_SIZE, IMG_SIZE, 3)):
    """Computes FLOPs (in GFLOPs) for a Keras model using the TF1-style profiler."""
    concrete_func = tf.function(lambda x: model(x))
    concrete_func = concrete_func.get_concrete_function(
        tf.TensorSpec(input_shape, model.inputs[0].dtype)
    )
    frozen_func = convert_variables_to_constants_v2(concrete_func)
    graph = frozen_func.graph

    run_meta = tf.compat.v1.RunMetadata()
    opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
    flops = tf.compat.v1.profiler.profile(
        graph=graph, run_meta=run_meta, cmd="op", options=opts
    )
    # profiler counts multiply+add as 2 ops -> divide by 2 for MACs, keep as FLOPs (common convention: report as-is /1e9)
    return flops.total_float_ops / 1e9


# ============================ SIZE / PARAM / TIMING PROFILER ============================
def profile_model(name, model, input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    print(f"\n--- Profiling: {name} ---")
    result = {"Model": name}

    # ---- Params ----
    total_params = model.count_params()
    trainable_params = int(np.sum([tf.size(w).numpy() for w in model.trainable_weights]))
    non_trainable_params = total_params - trainable_params
    result["Total_Params_M"] = round(total_params / 1e6, 3)
    result["Trainable_Params_M"] = round(trainable_params / 1e6, 3)
    result["Non_Trainable_Params_M"] = round(non_trainable_params / 1e6, 3)

    # ---- FLOPs ----
    try:
        flops_g = count_flops(model, input_shape=(1, *input_shape))
        result["FLOPs_G"] = round(flops_g, 3)
    except Exception as e:
        print(f"  [WARN] FLOPs calculation failed for {name}: {e}")
        result["FLOPs_G"] = np.nan

    # ---- Model size on disk ----
    try:
        if os.path.exists(TMP_SAVE_DIR):
            shutil.rmtree(TMP_SAVE_DIR)
        model.save(TMP_SAVE_DIR + ".keras")
        size_mb = os.path.getsize(TMP_SAVE_DIR + ".keras") / (1024 * 1024)
        os.remove(TMP_SAVE_DIR + ".keras")
        result["Model_Size_MB"] = round(size_mb, 2)
    except Exception as e:
        print(f"  [WARN] Save/size check failed for {name}: {e}")
        # fallback: estimate from param count assuming float32
        result["Model_Size_MB"] = round(total_params * 4 / (1024 * 1024), 2)

    # ---- Inference time / throughput ----
    try:
        dummy_input = tf.random.normal((BATCH_SIZE, *input_shape))

        # warm-up
        for _ in range(N_WARMUP):
            _ = model(dummy_input, training=False)

        times = []
        for _ in range(N_TIMED_RUNS):
            start = time.perf_counter()
            _ = model(dummy_input, training=False)
            times.append(time.perf_counter() - start)

        avg_time_s = np.mean(times)
        result["Inference_Time_ms"] = round(avg_time_s * 1000, 3)
        result["Throughput_img_per_sec"] = round(BATCH_SIZE / avg_time_s, 2)
    except Exception as e:
        print(f"  [WARN] Timing failed for {name}: {e}")
        result["Inference_Time_ms"] = np.nan
        result["Throughput_img_per_sec"] = np.nan

    print(f"  Params={result['Total_Params_M']}M  "
          f"FLOPs={result['FLOPs_G']}G  "
          f"Size={result['Model_Size_MB']}MB  "
          f"Latency={result['Inference_Time_ms']}ms")

    del model
    tf.keras.backend.clear_session()
    gc.collect()
    return result


# ============================ MODEL BUILDERS ============================
# Standard CNN backbones from tf.keras.applications -> classification head added.
def build_from_keras_app(app_fn, name, preprocess=None):
    def builder():
        # weights=None: complexity analysis (params/FLOPs/size/latency) only
        # depends on architecture, not on trained weight VALUES, so we skip
        # the ImageNet download entirely -> works with internet OFF on Kaggle.
        base = app_fn(include_top=False, weights=None,
                       input_shape=(IMG_SIZE, IMG_SIZE, 3))
        inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
        x = inputs if preprocess is None else preprocess(inputs)
        x = base(x, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(256, activation="relu")(x)
        outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
        return Model(inputs, outputs, name=name)
    return builder


MODEL_BUILDERS = {
    "MobileNetV2": build_from_keras_app(
        tf.keras.applications.MobileNetV2, "MobileNetV2",
        tf.keras.applications.mobilenet_v2.preprocess_input),

    "EfficientNetB0": build_from_keras_app(
        tf.keras.applications.EfficientNetB0, "EfficientNetB0",
        tf.keras.applications.efficientnet.preprocess_input),

    "ResNet50": build_from_keras_app(
        tf.keras.applications.ResNet50, "ResNet50",
        tf.keras.applications.resnet50.preprocess_input),

    "DenseNet121": build_from_keras_app(
        tf.keras.applications.DenseNet121, "DenseNet121",
        tf.keras.applications.densenet.preprocess_input),

    "VGG16": build_from_keras_app(
        tf.keras.applications.VGG16, "VGG16",
        tf.keras.applications.vgg16.preprocess_input),
}

# ---- ResNet18: not in tf.keras.applications, and no internet on Kaggle ----
# -> implemented here in pure Keras (basic-block ResNet18), no package/download needed.
def _resnet18_basic_block(x, filters, stride=1, downsample=False):
    shortcut = x
    x = layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, strides=1, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if downsample:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding="same", use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    return layers.ReLU()(x)

def build_resnet18():
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Conv2D(64, 7, strides=2, padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

    filters_per_stage = [64, 128, 256, 512]
    for stage_idx, filters in enumerate(filters_per_stage):
        for block_idx in range(2):                      # ResNet18 = 2 blocks/stage
            stride = 2 if (stage_idx > 0 and block_idx == 0) else 1
            downsample = (stride != 1) or (block_idx == 0 and stage_idx == 0 and filters != 64)
            x = _resnet18_basic_block(x, filters, stride=stride, downsample=downsample)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return Model(inputs, outputs, name="ResNet18")

MODEL_BUILDERS["ResNet18"] = build_resnet18


# ---- Vision Transformer: no internet -> minimal ViT-B/16-style implementation ----
# (patch embed + transformer encoder blocks), pure Keras, random init, no package needed.
class _ViTPatchEmbed(layers.Layer):
    def __init__(self, patch_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.proj = layers.Conv2D(embed_dim, kernel_size=patch_size, strides=patch_size)

    def call(self, x):
        x = self.proj(x)                       # (B, H/P, W/P, D)
        b = tf.shape(x)[0]
        return tf.reshape(x, (b, -1, x.shape[-1]))   # (B, N, D)

def _vit_transformer_block(x, embed_dim, num_heads, mlp_dim, drop=0.0):
    y = layers.LayerNormalization(epsilon=1e-6)(x)
    y = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads, dropout=drop)(y, y)
    x = layers.Add()([x, y])
    y = layers.LayerNormalization(epsilon=1e-6)(x)
    y = layers.Dense(mlp_dim, activation="gelu")(y)
    y = layers.Dense(embed_dim)(y)
    return layers.Add()([x, y])

def build_vit(patch_size=16, embed_dim=768, depth=12, num_heads=12, mlp_dim=3072):
    """ViT-B/16-equivalent architecture (params/FLOPs match the standard config)."""
    num_patches = (IMG_SIZE // patch_size) ** 2
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = _ViTPatchEmbed(patch_size, embed_dim)(inputs)

    cls_token = tf.Variable(tf.zeros((1, 1, embed_dim)), trainable=True, name="cls_token")
    pos_embed = tf.Variable(tf.random.normal((1, num_patches + 1, embed_dim), stddev=0.02),
                             trainable=True, name="pos_embed")

    def add_cls_and_pos(x):
        b = tf.shape(x)[0]
        cls = tf.broadcast_to(cls_token, (b, 1, embed_dim))
        x = tf.concat([cls, x], axis=1)
        return x + pos_embed

    x = layers.Lambda(add_cls_and_pos)(x)
    for _ in range(depth):
        x = _vit_transformer_block(x, embed_dim, num_heads, mlp_dim)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    cls_out = layers.Lambda(lambda t: t[:, 0])(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(cls_out)
    return Model(inputs, outputs, name="VisionTransformer")

MODEL_BUILDERS["VisionTransformer"] = build_vit

# ---- Swin Transformer: no official tf.keras.applications entry. ----
# Plug in your own Swin implementation/package here (e.g. a Kaggle dataset
# with weights, or `keras-cv`). Left as a placeholder so the rest of the
# script still runs.
def build_swin_placeholder():
    raise NotImplementedError(
        "Add your SwinTransformer builder here (custom implementation or package)."
    )
# MODEL_BUILDERS["SwinTransformer"] = build_swin_placeholder   # uncomment once implemented

# ---- SSL baselines: Deep Clustering / SwAV / VAE ----
# For complexity analysis, what matters at inference/deployment time is the
# ENCODER backbone each method uses (the SSL head/decoder is usually discarded
# after pretraining). Default below uses a ResNet50 encoder, which is the
# standard backbone for SwAV/DeepCluster in the literature -- swap in your
# actual encoder if yours differs.

def build_ssl_encoder(name):
    def builder():
        base = tf.keras.applications.ResNet50(
            include_top=False, weights=None,
            input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )
        inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
        x = tf.keras.applications.resnet50.preprocess_input(inputs)
        x = base(x, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(256, activation="relu")(x)
        outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
        return Model(inputs, outputs, name=name)
    return builder

MODEL_BUILDERS["Deep Clustering"] = build_ssl_encoder("Deep_Clustering")
MODEL_BUILDERS["SwAV"] = build_ssl_encoder("SwAV")

# ---- VAE: encoder-only complexity (decoder isn't used at inference/classification time) ----
def build_vae_encoder():
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Conv2D(32, 3, strides=2, padding="same", activation="relu")(inputs)
    x = layers.Conv2D(64, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Conv2D(128, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Conv2D(256, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return Model(inputs, outputs, name="VAE_Encoder")
    # NOTE: replace with your actual VAE encoder architecture if different.

MODEL_BUILDERS["VAE"] = build_vae_encoder

# ---- YOUR PROPOSED MODEL (SSL-pretrained encoder + CBAM head) ----
# Your CBAM block uses these two custom layers (copied from your training
# notebook) -- Keras needs them explicitly to deserialize the saved model.
class ChannelMean(layers.Layer):
    def call(self, x):
        return tf.reduce_mean(x, axis=-1, keepdims=True)

class ChannelMax(layers.Layer):
    def call(self, x):
        return tf.reduce_max(x, axis=-1, keepdims=True)

CBAM_CUSTOM_OBJECTS = {"ChannelMean": ChannelMean, "ChannelMax": ChannelMax}

PROPOSED_MODEL_PATH = "/kaggle/working/AgriSSL_CBAMNet_labdata.keras"
# ^ matches your Kaggle output. If this isn't the right one, your folder also
#   has "best_model.keras" -- swap the path to that instead if needed:
#   PROPOSED_MODEL_PATH = "/kaggle/working/best_model.keras"

def build_proposed_model():
    if not os.path.exists(PROPOSED_MODEL_PATH):
        raise FileNotFoundError(
            f"Proposed model not found at: {PROPOSED_MODEL_PATH}\n"
            f"-> Edit PROPOSED_MODEL_PATH above to point to your saved .keras file."
        )
    return tf.keras.models.load_model(
        PROPOSED_MODEL_PATH, compile=False, custom_objects=CBAM_CUSTOM_OBJECTS
    )

MODEL_BUILDERS["Proposed Model"] = build_proposed_model


# ============================ MAIN LOOP ============================
if os.path.exists(CSV_PATH):
    results_df = pd.read_csv(CSV_PATH)
    done_models = set(results_df["Model"].tolist())
    print(f"Resuming... {len(done_models)} models already done")
else:
    results_df = pd.DataFrame(columns=result_columns)
    done_models = set()

import traceback

for name, builder in MODEL_BUILDERS.items():
    if name in done_models:
        print(f"Skipping (already done): {name}")
        continue
    try:
        model = builder()
        row = profile_model(name, model)
        results_df.loc[len(results_df)] = [row.get(c, np.nan) for c in result_columns]
        results_df.to_csv(CSV_PATH, index=False)
    except NotImplementedError as e:
        print(f"[SKIP] {name}: {e}")
    except Exception as e:
        print(f"[ERROR] {name} failed: {e}")
        traceback.print_exc()

print("\n================================================================")
print("MODEL COMPLEXITY COMPARISON")
print("================================================================")
print(results_df)
print(f"\nSaved: {CSV_PATH}")"""
========================================================================
 MODEL COMPLEXITY COMPARISON SCRIPT
 Computes: Params, Trainable Params, FLOPs, Model Size, Inference Time,
           Throughput  -- for every baseline + the proposed model.
 (No accuracy column -- pure complexity/efficiency analysis)

 Run this on Kaggle. Output -> model_complexity_comparison.csv
========================================================================
"""

import os
import gc
import time
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model

from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

print("TensorFlow:", tf.__version__)

# ============================ CONFIG ============================
IMG_SIZE     = 224
NUM_CLASSES  = 5          # <-- change to your actual number of classes
N_WARMUP     = 5          # warm-up runs before timing (excluded from avg)
N_TIMED_RUNS = 50         # runs actually used to measure inference time
BATCH_SIZE   = 1          # per-image latency; change to measure batch throughput
CSV_PATH     = "sabbir-final-csv-modelcomplexity.csv"
TMP_SAVE_DIR = "tmp_model_check"

result_columns = [
    "Model", "Total_Params_M", "Trainable_Params_M", "Non_Trainable_Params_M",
    "FLOPs_G", "Model_Size_MB", "Inference_Time_ms", "Throughput_img_per_sec"
]

# ============================ FLOPs COUNTER ============================
def count_flops(model, input_shape=(1, IMG_SIZE, IMG_SIZE, 3)):
    """Computes FLOPs (in GFLOPs) for a Keras model using the TF1-style profiler."""
    concrete_func = tf.function(lambda x: model(x))
    concrete_func = concrete_func.get_concrete_function(
        tf.TensorSpec(input_shape, model.inputs[0].dtype)
    )
    frozen_func = convert_variables_to_constants_v2(concrete_func)
    graph = frozen_func.graph

    run_meta = tf.compat.v1.RunMetadata()
    opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
    flops = tf.compat.v1.profiler.profile(
        graph=graph, run_meta=run_meta, cmd="op", options=opts
    )
    # profiler counts multiply+add as 2 ops -> divide by 2 for MACs, keep as FLOPs (common convention: report as-is /1e9)
    return flops.total_float_ops / 1e9


# ============================ SIZE / PARAM / TIMING PROFILER ============================
def profile_model(name, model, input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    print(f"\n--- Profiling: {name} ---")
    result = {"Model": name}

    # ---- Params ----
    total_params = model.count_params()
    trainable_params = int(np.sum([tf.size(w).numpy() for w in model.trainable_weights]))
    non_trainable_params = total_params - trainable_params
    result["Total_Params_M"] = round(total_params / 1e6, 3)
    result["Trainable_Params_M"] = round(trainable_params / 1e6, 3)
    result["Non_Trainable_Params_M"] = round(non_trainable_params / 1e6, 3)

    # ---- FLOPs ----
    try:
        flops_g = count_flops(model, input_shape=(1, *input_shape))
        result["FLOPs_G"] = round(flops_g, 3)
    except Exception as e:
        print(f"  [WARN] FLOPs calculation failed for {name}: {e}")
        result["FLOPs_G"] = np.nan

    # ---- Model size on disk ----
    try:
        if os.path.exists(TMP_SAVE_DIR):
            shutil.rmtree(TMP_SAVE_DIR)
        model.save(TMP_SAVE_DIR + ".keras")
        size_mb = os.path.getsize(TMP_SAVE_DIR + ".keras") / (1024 * 1024)
        os.remove(TMP_SAVE_DIR + ".keras")
        result["Model_Size_MB"] = round(size_mb, 2)
    except Exception as e:
        print(f"  [WARN] Save/size check failed for {name}: {e}")
        # fallback: estimate from param count assuming float32
        result["Model_Size_MB"] = round(total_params * 4 / (1024 * 1024), 2)

    # ---- Inference time / throughput ----
    try:
        dummy_input = tf.random.normal((BATCH_SIZE, *input_shape))

        # warm-up
        for _ in range(N_WARMUP):
            _ = model(dummy_input, training=False)

        times = []
        for _ in range(N_TIMED_RUNS):
            start = time.perf_counter()
            _ = model(dummy_input, training=False)
            times.append(time.perf_counter() - start)

        avg_time_s = np.mean(times)
        result["Inference_Time_ms"] = round(avg_time_s * 1000, 3)
        result["Throughput_img_per_sec"] = round(BATCH_SIZE / avg_time_s, 2)
    except Exception as e:
        print(f"  [WARN] Timing failed for {name}: {e}")
        result["Inference_Time_ms"] = np.nan
        result["Throughput_img_per_sec"] = np.nan

    print(f"  Params={result['Total_Params_M']}M  "
          f"FLOPs={result['FLOPs_G']}G  "
          f"Size={result['Model_Size_MB']}MB  "
          f"Latency={result['Inference_Time_ms']}ms")

    del model
    tf.keras.backend.clear_session()
    gc.collect()
    return result


# ============================ MODEL BUILDERS ============================
# Standard CNN backbones from tf.keras.applications -> classification head added.
def build_from_keras_app(app_fn, name, preprocess=None):
    def builder():
        # weights=None: complexity analysis (params/FLOPs/size/latency) only
        # depends on architecture, not on trained weight VALUES, so we skip
        # the ImageNet download entirely -> works with internet OFF on Kaggle.
        base = app_fn(include_top=False, weights=None,
                       input_shape=(IMG_SIZE, IMG_SIZE, 3))
        inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
        x = inputs if preprocess is None else preprocess(inputs)
        x = base(x, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(256, activation="relu")(x)
        outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
        return Model(inputs, outputs, name=name)
    return builder


MODEL_BUILDERS = {
    "MobileNetV2": build_from_keras_app(
        tf.keras.applications.MobileNetV2, "MobileNetV2",
        tf.keras.applications.mobilenet_v2.preprocess_input),

    "EfficientNetB0": build_from_keras_app(
        tf.keras.applications.EfficientNetB0, "EfficientNetB0",
        tf.keras.applications.efficientnet.preprocess_input),

    "ResNet50": build_from_keras_app(
        tf.keras.applications.ResNet50, "ResNet50",
        tf.keras.applications.resnet50.preprocess_input),

    "DenseNet121": build_from_keras_app(
        tf.keras.applications.DenseNet121, "DenseNet121",
        tf.keras.applications.densenet.preprocess_input),

    "VGG16": build_from_keras_app(
        tf.keras.applications.VGG16, "VGG16",
        tf.keras.applications.vgg16.preprocess_input),
}

# ---- ResNet18: not in tf.keras.applications, and no internet on Kaggle ----
# -> implemented here in pure Keras (basic-block ResNet18), no package/download needed.
def _resnet18_basic_block(x, filters, stride=1, downsample=False):
    shortcut = x
    x = layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, strides=1, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if downsample:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding="same", use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    return layers.ReLU()(x)

def build_resnet18():
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Conv2D(64, 7, strides=2, padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

    filters_per_stage = [64, 128, 256, 512]
    for stage_idx, filters in enumerate(filters_per_stage):
        for block_idx in range(2):                      # ResNet18 = 2 blocks/stage
            stride = 2 if (stage_idx > 0 and block_idx == 0) else 1
            downsample = (stride != 1) or (block_idx == 0 and stage_idx == 0 and filters != 64)
            x = _resnet18_basic_block(x, filters, stride=stride, downsample=downsample)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return Model(inputs, outputs, name="ResNet18")

MODEL_BUILDERS["ResNet18"] = build_resnet18


# ---- Vision Transformer: no internet -> minimal ViT-B/16-style implementation ----
# (patch embed + transformer encoder blocks), pure Keras, random init, no package needed.
class _ViTPatchEmbed(layers.Layer):
    def __init__(self, patch_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.proj = layers.Conv2D(embed_dim, kernel_size=patch_size, strides=patch_size)

    def call(self, x):
        x = self.proj(x)                       # (B, H/P, W/P, D)
        b = tf.shape(x)[0]
        return tf.reshape(x, (b, -1, x.shape[-1]))   # (B, N, D)

def _vit_transformer_block(x, embed_dim, num_heads, mlp_dim, drop=0.0):
    y = layers.LayerNormalization(epsilon=1e-6)(x)
    y = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads, dropout=drop)(y, y)
    x = layers.Add()([x, y])
    y = layers.LayerNormalization(epsilon=1e-6)(x)
    y = layers.Dense(mlp_dim, activation="gelu")(y)
    y = layers.Dense(embed_dim)(y)
    return layers.Add()([x, y])

def build_vit(patch_size=16, embed_dim=768, depth=12, num_heads=12, mlp_dim=3072):
    """ViT-B/16-equivalent architecture (params/FLOPs match the standard config)."""
    num_patches = (IMG_SIZE // patch_size) ** 2
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = _ViTPatchEmbed(patch_size, embed_dim)(inputs)

    cls_token = tf.Variable(tf.zeros((1, 1, embed_dim)), trainable=True, name="cls_token")
    pos_embed = tf.Variable(tf.random.normal((1, num_patches + 1, embed_dim), stddev=0.02),
                             trainable=True, name="pos_embed")

    def add_cls_and_pos(x):
        b = tf.shape(x)[0]
        cls = tf.broadcast_to(cls_token, (b, 1, embed_dim))
        x = tf.concat([cls, x], axis=1)
        return x + pos_embed

    x = layers.Lambda(add_cls_and_pos)(x)
    for _ in range(depth):
        x = _vit_transformer_block(x, embed_dim, num_heads, mlp_dim)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    cls_out = layers.Lambda(lambda t: t[:, 0])(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(cls_out)
    return Model(inputs, outputs, name="VisionTransformer")

MODEL_BUILDERS["VisionTransformer"] = build_vit

# ---- Swin Transformer: no official tf.keras.applications entry. ----
# Plug in your own Swin implementation/package here (e.g. a Kaggle dataset
# with weights, or `keras-cv`). Left as a placeholder so the rest of the
# script still runs.
def build_swin_placeholder():
    raise NotImplementedError(
        "Add your SwinTransformer builder here (custom implementation or package)."
    )
# MODEL_BUILDERS["SwinTransformer"] = build_swin_placeholder   # uncomment once implemented

# ---- SSL baselines: Deep Clustering / SwAV / VAE ----
# For complexity analysis, what matters at inference/deployment time is the
# ENCODER backbone each method uses (the SSL head/decoder is usually discarded
# after pretraining). Default below uses a ResNet50 encoder, which is the
# standard backbone for SwAV/DeepCluster in the literature -- swap in your
# actual encoder if yours differs.

def build_ssl_encoder(name):
    def builder():
        base = tf.keras.applications.ResNet50(
            include_top=False, weights=None,
            input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )
        inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
        x = tf.keras.applications.resnet50.preprocess_input(inputs)
        x = base(x, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(256, activation="relu")(x)
        outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
        return Model(inputs, outputs, name=name)
    return builder

MODEL_BUILDERS["Deep Clustering"] = build_ssl_encoder("Deep_Clustering")
MODEL_BUILDERS["SwAV"] = build_ssl_encoder("SwAV")

# ---- VAE: encoder-only complexity (decoder isn't used at inference/classification time) ----
def build_vae_encoder():
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Conv2D(32, 3, strides=2, padding="same", activation="relu")(inputs)
    x = layers.Conv2D(64, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Conv2D(128, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Conv2D(256, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return Model(inputs, outputs, name="VAE_Encoder")
    # NOTE: replace with your actual VAE encoder architecture if different.

MODEL_BUILDERS["VAE"] = build_vae_encoder

# ---- YOUR PROPOSED MODEL (SSL-pretrained encoder + CBAM head) ----
# Your CBAM block uses these two custom layers (copied from your training
# notebook) -- Keras needs them explicitly to deserialize the saved model.
class ChannelMean(layers.Layer):
    def call(self, x):
        return tf.reduce_mean(x, axis=-1, keepdims=True)

class ChannelMax(layers.Layer):
    def call(self, x):
        return tf.reduce_max(x, axis=-1, keepdims=True)

CBAM_CUSTOM_OBJECTS = {"ChannelMean": ChannelMean, "ChannelMax": ChannelMax}

PROPOSED_MODEL_PATH = "/kaggle/working/AgriSSL_CBAMNet_labdata.keras"
# ^ matches your Kaggle output. If this isn't the right one, your folder also
#   has "best_model.keras" -- swap the path to that instead if needed:
#   PROPOSED_MODEL_PATH = "/kaggle/working/best_model.keras"

def build_proposed_model():
    if not os.path.exists(PROPOSED_MODEL_PATH):
        raise FileNotFoundError(
            f"Proposed model not found at: {PROPOSED_MODEL_PATH}\n"
            f"-> Edit PROPOSED_MODEL_PATH above to point to your saved .keras file."
        )
    return tf.keras.models.load_model(
        PROPOSED_MODEL_PATH, compile=False, custom_objects=CBAM_CUSTOM_OBJECTS
    )

MODEL_BUILDERS["Proposed Model"] = build_proposed_model


# ============================ MAIN LOOP ============================
if os.path.exists(CSV_PATH):
    results_df = pd.read_csv(CSV_PATH)
    done_models = set(results_df["Model"].tolist())
    print(f"Resuming... {len(done_models)} models already done")
else:
    results_df = pd.DataFrame(columns=result_columns)
    done_models = set()

import traceback

for name, builder in MODEL_BUILDERS.items():
    if name in done_models:
        print(f"Skipping (already done): {name}")
        continue
    try:
        model = builder()
        row = profile_model(name, model)
        results_df.loc[len(results_df)] = [row.get(c, np.nan) for c in result_columns]
        results_df.to_csv(CSV_PATH, index=False)
    except NotImplementedError as e:
        print(f"[SKIP] {name}: {e}")
    except Exception as e:
        print(f"[ERROR] {name} failed: {e}")
        traceback.print_exc()

print("\n================================================================")
print("MODEL COMPLEXITY COMPARISON")
print("================================================================")
print(results_df)
print(f"\nSaved: {CSV_PATH}")

In [ ]:
"""
========================================================================
 MODEL COMPLEXITY COMPARISON SCRIPT
 Computes: Params, Trainable Params, FLOPs, Model Size, Inference Time,
           Throughput  -- for every baseline + the proposed model.
 (No accuracy column -- pure complexity/efficiency analysis)

 Run this on Kaggle. Output -> model_complexity_comparison.csv
========================================================================
"""

import os
import gc
import time
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model

from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

print("TensorFlow:", tf.__version__)

# ============================ CONFIG ============================
IMG_SIZE     = 224
NUM_CLASSES  = 5          # <-- change to your actual number of classes
N_WARMUP     = 5          # warm-up runs before timing (excluded from avg)
N_TIMED_RUNS = 50         # runs actually used to measure inference time
BATCH_SIZE   = 1          # per-image latency; change to measure batch throughput
CSV_PATH     = "sabbir-final-csv-modelcomplexity.csv"
TMP_SAVE_DIR = "tmp_model_check"

result_columns = [
    "Model", "Total_Params_M", "Trainable_Params_M", "Non_Trainable_Params_M",
    "FLOPs_G", "Model_Size_MB", "Inference_Time_ms", "Throughput_img_per_sec"
]

# ============================ FLOPs COUNTER ============================
def count_flops(model, input_shape=(1, IMG_SIZE, IMG_SIZE, 3)):
    """Computes FLOPs (in GFLOPs) for a Keras model using the TF1-style profiler."""
    concrete_func = tf.function(lambda x: model(x))
    concrete_func = concrete_func.get_concrete_function(
        tf.TensorSpec(input_shape, model.inputs[0].dtype)
    )
    frozen_func = convert_variables_to_constants_v2(concrete_func)
    graph = frozen_func.graph

    run_meta = tf.compat.v1.RunMetadata()
    opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
    flops = tf.compat.v1.profiler.profile(
        graph=graph, run_meta=run_meta, cmd="op", options=opts
    )
    # profiler counts multiply+add as 2 ops -> divide by 2 for MACs, keep as FLOPs (common convention: report as-is /1e9)
    return flops.total_float_ops / 1e9


# ============================ SIZE / PARAM / TIMING PROFILER ============================
def profile_model(name, model, input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    print(f"\n--- Profiling: {name} ---")
    result = {"Model": name}

    # ---- Params ----
    total_params = model.count_params()
    trainable_params = int(np.sum([tf.size(w).numpy() for w in model.trainable_weights]))
    non_trainable_params = total_params - trainable_params
    result["Total_Params_M"] = round(total_params / 1e6, 3)
    result["Trainable_Params_M"] = round(trainable_params / 1e6, 3)
    result["Non_Trainable_Params_M"] = round(non_trainable_params / 1e6, 3)

    # ---- FLOPs ----
    try:
        flops_g = count_flops(model, input_shape=(1, *input_shape))
        result["FLOPs_G"] = round(flops_g, 3)
    except Exception as e:
        print(f"  [WARN] FLOPs calculation failed for {name}: {e}")
        result["FLOPs_G"] = np.nan

    # ---- Model size on disk ----
    try:
        if os.path.exists(TMP_SAVE_DIR):
            shutil.rmtree(TMP_SAVE_DIR)
        model.save(TMP_SAVE_DIR + ".keras")
        size_mb = os.path.getsize(TMP_SAVE_DIR + ".keras") / (1024 * 1024)
        os.remove(TMP_SAVE_DIR + ".keras")
        result["Model_Size_MB"] = round(size_mb, 2)
    except Exception as e:
        print(f"  [WARN] Save/size check failed for {name}: {e}")
        # fallback: estimate from param count assuming float32
        result["Model_Size_MB"] = round(total_params * 4 / (1024 * 1024), 2)

    # ---- Inference time / throughput ----
    try:
        dummy_input = tf.random.normal((BATCH_SIZE, *input_shape))

        # warm-up
        for _ in range(N_WARMUP):
            _ = model(dummy_input, training=False)

        times = []
        for _ in range(N_TIMED_RUNS):
            start = time.perf_counter()
            _ = model(dummy_input, training=False)
            times.append(time.perf_counter() - start)

        avg_time_s = np.mean(times)
        result["Inference_Time_ms"] = round(avg_time_s * 1000, 3)
        result["Throughput_img_per_sec"] = round(BATCH_SIZE / avg_time_s, 2)
    except Exception as e:
        print(f"  [WARN] Timing failed for {name}: {e}")
        result["Inference_Time_ms"] = np.nan
        result["Throughput_img_per_sec"] = np.nan

    print(f"  Params={result['Total_Params_M']}M  "
          f"FLOPs={result['FLOPs_G']}G  "
          f"Size={result['Model_Size_MB']}MB  "
          f"Latency={result['Inference_Time_ms']}ms")

    del model
    tf.keras.backend.clear_session()
    gc.collect()
    return result


# ============================ MODEL BUILDERS ============================
# Standard CNN backbones from tf.keras.applications -> classification head added.
def build_from_keras_app(app_fn, name, preprocess=None):
    def builder():
        # weights=None: complexity analysis (params/FLOPs/size/latency) only
        # depends on architecture, not on trained weight VALUES, so we skip
        # the ImageNet download entirely -> works with internet OFF on Kaggle.
        base = app_fn(include_top=False, weights=None,
                       input_shape=(IMG_SIZE, IMG_SIZE, 3))
        inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
        x = inputs if preprocess is None else preprocess(inputs)
        x = base(x, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(256, activation="relu")(x)
        outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
        return Model(inputs, outputs, name=name)
    return builder


MODEL_BUILDERS = {
    "MobileNetV2": build_from_keras_app(
        tf.keras.applications.MobileNetV2, "MobileNetV2",
        tf.keras.applications.mobilenet_v2.preprocess_input),

    "EfficientNetB0": build_from_keras_app(
        tf.keras.applications.EfficientNetB0, "EfficientNetB0",
        tf.keras.applications.efficientnet.preprocess_input),

    "ResNet50": build_from_keras_app(
        tf.keras.applications.ResNet50, "ResNet50",
        tf.keras.applications.resnet50.preprocess_input),

    "DenseNet121": build_from_keras_app(
        tf.keras.applications.DenseNet121, "DenseNet121",
        tf.keras.applications.densenet.preprocess_input),

    "VGG16": build_from_keras_app(
        tf.keras.applications.VGG16, "VGG16",
        tf.keras.applications.vgg16.preprocess_input),
}

# ---- ResNet18: not in tf.keras.applications, and no internet on Kaggle ----
# -> implemented here in pure Keras (basic-block ResNet18), no package/download needed.
def _resnet18_basic_block(x, filters, stride=1, downsample=False):
    shortcut = x
    x = layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, strides=1, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if downsample:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding="same", use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    return layers.ReLU()(x)

def build_resnet18():
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Conv2D(64, 7, strides=2, padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

    filters_per_stage = [64, 128, 256, 512]
    for stage_idx, filters in enumerate(filters_per_stage):
        for block_idx in range(2):                      # ResNet18 = 2 blocks/stage
            stride = 2 if (stage_idx > 0 and block_idx == 0) else 1
            downsample = (stride != 1) or (block_idx == 0 and stage_idx == 0 and filters != 64)
            x = _resnet18_basic_block(x, filters, stride=stride, downsample=downsample)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return Model(inputs, outputs, name="ResNet18")

MODEL_BUILDERS["ResNet18"] = build_resnet18


# ---- Vision Transformer: no internet -> minimal ViT-B/16-style implementation ----
# (patch embed + transformer encoder blocks), pure Keras, random init, no package needed.
class _ViTPatchEmbed(layers.Layer):
    def __init__(self, patch_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.proj = layers.Conv2D(embed_dim, kernel_size=patch_size, strides=patch_size)

    def call(self, x):
        x = self.proj(x)                       # (B, H/P, W/P, D)
        b = tf.shape(x)[0]
        return tf.reshape(x, (b, -1, x.shape[-1]))   # (B, N, D)

def _vit_transformer_block(x, embed_dim, num_heads, mlp_dim, drop=0.0):
    y = layers.LayerNormalization(epsilon=1e-6)(x)
    y = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads, dropout=drop)(y, y)
    x = layers.Add()([x, y])
    y = layers.LayerNormalization(epsilon=1e-6)(x)
    y = layers.Dense(mlp_dim, activation="gelu")(y)
    y = layers.Dense(embed_dim)(y)
    return layers.Add()([x, y])

def build_vit(patch_size=16, embed_dim=768, depth=12, num_heads=12, mlp_dim=3072):
    """ViT-B/16-equivalent architecture (params/FLOPs match the standard config)."""
    num_patches = (IMG_SIZE // patch_size) ** 2
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = _ViTPatchEmbed(patch_size, embed_dim)(inputs)

    cls_token = tf.Variable(tf.zeros((1, 1, embed_dim)), trainable=True, name="cls_token")
    pos_embed = tf.Variable(tf.random.normal((1, num_patches + 1, embed_dim), stddev=0.02),
                             trainable=True, name="pos_embed")

    def add_cls_and_pos(x):
        b = tf.shape(x)[0]
        cls = tf.broadcast_to(cls_token, (b, 1, embed_dim))
        x = tf.concat([cls, x], axis=1)
        return x + pos_embed

    x = layers.Lambda(add_cls_and_pos)(x)
    for _ in range(depth):
        x = _vit_transformer_block(x, embed_dim, num_heads, mlp_dim)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    cls_out = layers.Lambda(lambda t: t[:, 0])(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(cls_out)
    return Model(inputs, outputs, name="VisionTransformer")

MODEL_BUILDERS["VisionTransformer"] = build_vit

# ---- Swin Transformer: no official tf.keras.applications entry. ----
# Plug in your own Swin implementation/package here (e.g. a Kaggle dataset
# with weights, or `keras-cv`). Left as a placeholder so the rest of the
# script still runs.
def build_swin_placeholder():
    raise NotImplementedError(
        "Add your SwinTransformer builder here (custom implementation or package)."
    )
# MODEL_BUILDERS["SwinTransformer"] = build_swin_placeholder   # uncomment once implemented

# ---- SSL baselines: Deep Clustering / SwAV / VAE ----
# For complexity analysis, what matters at inference/deployment time is the
# ENCODER backbone each method uses (the SSL head/decoder is usually discarded
# after pretraining). Default below uses a ResNet50 encoder, which is the
# standard backbone for SwAV/DeepCluster in the literature -- swap in your
# actual encoder if yours differs.

def build_ssl_encoder(name):
    def builder():
        base = tf.keras.applications.ResNet50(
            include_top=False, weights=None,
            input_shape=(IMG_SIZE, IMG_SIZE, 3)
        )
        inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
        x = tf.keras.applications.resnet50.preprocess_input(inputs)
        x = base(x, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(256, activation="relu")(x)
        outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
        return Model(inputs, outputs, name=name)
    return builder

MODEL_BUILDERS["Deep Clustering"] = build_ssl_encoder("Deep_Clustering")
MODEL_BUILDERS["SwAV"] = build_ssl_encoder("SwAV")

# ---- VAE: encoder-only complexity (decoder isn't used at inference/classification time) ----
def build_vae_encoder():
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Conv2D(32, 3, strides=2, padding="same", activation="relu")(inputs)
    x = layers.Conv2D(64, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Conv2D(128, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Conv2D(256, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return Model(inputs, outputs, name="VAE_Encoder")
    # NOTE: replace with your actual VAE encoder architecture if different.

MODEL_BUILDERS["VAE"] = build_vae_encoder

# ---- YOUR PROPOSED MODEL (SSL-pretrained encoder + CBAM head) ----
PROPOSED_MODEL_PATH = "/kaggle/working/AgriSSL_CBAMNet_labdata.keras"
# ^ matches your Kaggle output. If this isn't the right one, your folder also
#   has "best_model.keras" -- swap the path to that instead if needed:
#   PROPOSED_MODEL_PATH = "/kaggle/working/best_model.keras"

def build_proposed_model():
    if not os.path.exists(PROPOSED_MODEL_PATH):
        raise FileNotFoundError(
            f"Proposed model not found at: {PROPOSED_MODEL_PATH}\n"
            f"-> Edit PROPOSED_MODEL_PATH above to point to your saved .keras file."
        )
    return tf.keras.models.load_model(PROPOSED_MODEL_PATH, compile=False)

MODEL_BUILDERS["Proposed Model"] = build_proposed_model


# ============================ MAIN LOOP ============================
if os.path.exists(CSV_PATH):
    results_df = pd.read_csv(CSV_PATH)
    done_models = set(results_df["Model"].tolist())
    print(f"Resuming... {len(done_models)} models already done")
else:
    results_df = pd.DataFrame(columns=result_columns)
    done_models = set()

for name, builder in MODEL_BUILDERS.items():
    if name in done_models:
        print(f"Skipping (already done): {name}")
        continue
    try:
        model = builder()
        row = profile_model(name, model)
        results_df.loc[len(results_df)] = [row.get(c, np.nan) for c in result_columns]
        results_df.to_csv(CSV_PATH, index=False)
    except NotImplementedError as e:
        print(f"[SKIP] {name}: {e}")
    except Exception as e:
        print(f"[ERROR] {name} failed: {e}")

print("\n================================================================")
print("MODEL COMPLEXITY COMPARISON")
print("================================================================")
print(results_df)
print(f"\nSaved: {CSV_PATH}")